# NeuroAD Discovery Engine — Hypothesis-First Investigation Walkthrough

**A reproducible, fully offline submission notebook.**

This notebook drives the engine the way a researcher actually would: you start
with a *plain-language hunch*, the harness turns it into a **pre-registered,
falsifiable spec** *before any math runs*, and only then does the five-test
referee gauntlet get to render a verdict. The point of the whole instrument is
**calibrated honesty** — it is designed to *kill* its own findings when they
don't survive scrutiny, and to stop short of ever calling a candidate a proven
biomarker.

We tell a **two-act story**:

- **Act I — a SURVIVOR.** A hypothesis that clears the confound gauntlet *and*
  the molecular anchor gate, and is promoted to a follow-up candidate.
- **Act II — a KILL.** A hypothesis whose signal is a **scanner artifact**; the
  referee honestly fails it and blocks promotion.

Every cell runs **offline and deterministically** — no `ANTHROPIC_API_KEY`, no
network. The single entry point is
`neuroad.harness.orchestrator.investigate(hypothesis, dataset)`.

> Run with: `PYTHONPATH=src python -m jupyter nbconvert --to notebook --execute investigate_walkthrough.ipynb`


In [ ]:
# --- Offline, deterministic setup -------------------------------------------
# Force the fully-offline path: no API key means investigate() uses its
# deterministic keyword parser and hardcoded policy fallbacks end-to-end.
import os, json, textwrap
os.environ.pop("ANTHROPIC_API_KEY", None)

from neuroad.harness.orchestrator import investigate

def show(title, lines):
    bar = "=" * 74
    print(bar); print(title); print(bar)
    for ln in lines:
        print(ln)

print("Environment: ANTHROPIC_API_KEY present?",
      "ANTHROPIC_API_KEY" in os.environ, "(False = fully offline)")


## Act I — The SURVIVOR

### 1. A researcher's plain-language hypothesis

No structured input, no schema, no config. Just a sentence a domain scientist
would say out loud:


In [ ]:
hypothesis = (
    "Structural MRI embeddings separate MCI converters from non-converters"
)
print("Researcher hypothesis:\n  " + hypothesis)


### 2. The engine PRE-REGISTERS a spec — *before* it sees any results

This is the heart of the hypothesis-first workflow. Calling `investigate` first
**parses the hunch into a structured claim**, then **enriches it from the policy
layer** with three things that are *locked in before the experiment runs*:

- a **novelty_class** (how much prior art this contrast has),
- a **pre-registered expected_direction** (what a real effect should look like),
- a **pre-registered kill_criterion** (the number that would falsify it).

Also stamped up front: the **honesty rung** the finding is allowed to claim —
one step on a 5-rung ladder that *never* reaches "proven biomarker".

We run the call, but in this cell we deliberately look **only at the
pre-registered spec** — the falsifiers a referee fixes *before* peeking at the
data.


In [ ]:
card_survivor = investigate(hypothesis, "synthetic:SURVIVOR")
d = card_survivor.to_dict()
prov = d["discovery_provenance"]

show("PRE-REGISTERED SPEC  (fixed BEFORE any result is read)", [
    f"claim_text        : {d['claim_text']}",
    f"substrate         : {d['substrate']}",
    f"head              : {d['head']}",
    f"population         : {d['population']['group_a']}  vs  {d['population']['group_b']}",
    "",
    f"novelty_class     : {card_survivor.novelty_class}",
    f"routed_mechanism  : {prov['anchor_gate']['routed_mechanism']}",
    "",
    "PRE-REGISTERED expected_direction (what a REAL effect must look like):",
    textwrap.fill(prov["expected_direction"], 70, initial_indent="  ", subsequent_indent="  "),
    "",
    "PRE-REGISTERED kill_criterion (the number that would FALSIFY it):",
    textwrap.fill(prov["kill_criterion"], 70, initial_indent="  ", subsequent_indent="  "),
    "",
    f"honesty_rung CLAIMED : {card_survivor.honesty_rung}",
    "   (top of a 5-rung ladder that never asserts 'proven biomarker')",
])


### 3. NOW the referee speaks — gauntlet verdict, biomarker anchor, experiment card

Only after the falsifiers are fixed do we read the results. The five-test
gauntlet challenges the naive effect against age/sex, scanner/site leakage, a
brain-age proxy, a molecular anchor, and held-out replication. The
**biomarker-anchor gate** is a hard gate: a promoted finding whose plasma
p-tau217 / GFAP correlate fails is blocked.


In [ ]:
ne = d["naive_effect"]
gate = prov["anchor_gate"]

show("ACT I RESULT  — the referee's verdict", [
    f"naive effect       : {ne['metric']} = {ne['value']}  (n={ne.get('n')})",
    "",
    "FIVE-TEST GAUNTLET:",
    *[f"   {k:<16}: {v.upper()}" for k, v in d["robustness"].items()],
    "",
    f"robustness_score   : {d['robustness_score']} / 100",
    f"verdict            : {d['verdict'].upper()}",
    f"promoted           : {d['promoted']}",
    "",
    "BIOMARKER ANCHOR (molecular hard gate):",
    f"   status          : {gate['status'].upper()}",
    f"   blocked_promotion: {gate['blocked_promotion']}",
    f"   T p-tau217 r     : {d['atn_profile'].get('T_ptau217_r')}"
    f"  (CI lo {d['atn_profile'].get('T_ptau217_ci_lo')})",
])


In [ ]:
# The researcher-facing EXPERIMENT CARD: the named next experiment + biology.
show("ACT I  — EXPERIMENT CARD", [
    "biology_hypothesis:",
    textwrap.fill(d["biology_hypothesis"], 72, initial_indent="  ", subsequent_indent="  "),
    "",
    "next_experiment (the concrete, pre-registered follow-up):",
    *[f"   - {step}" for step in d["next_experiment"]],
])
assert d["promoted"] is True, "Act I should SURVIVE and be promoted"
assert card_survivor.honesty_rung == "severity_anchored"
print("\nAct I check: SURVIVOR promoted, capped at 'severity_anchored' (NOT 'proven'). OK")


## Act II — The KILL (the referee is honest)

Same instrument, same hypothesis text, a different cohort:
`synthetic:KILL`. Here the apparent separability is driven by a **scanner /
site artifact** — the kind of leak that has embarrassed brain-foundation-model
papers. A credulous pipeline would report a strong AUC. Our referee is built to
**catch and kill it**.

Again we look at the **pre-registered spec first**, then the verdict.


In [ ]:
card_kill = investigate(hypothesis, "synthetic:KILL")
k = card_kill.to_dict()
kprov = k["discovery_provenance"]

show("ACT II  — PRE-REGISTERED SPEC (identical falsifiers, fixed up front)", [
    f"novelty_class     : {card_kill.novelty_class}",
    f"routed_mechanism  : {kprov['anchor_gate']['routed_mechanism']}",
    "kill_criterion (locked BEFORE results):",
    textwrap.fill(kprov["kill_criterion"], 70, initial_indent="  ", subsequent_indent="  "),
])


In [ ]:
kne = k["naive_effect"]
kgate = kprov["anchor_gate"]

show("ACT II RESULT  — the referee KILLS a scanner artifact", [
    f"naive effect       : {kne['metric']} = {kne['value']}  (n={kne.get('n')})",
    "   ^ looks strong at face value — a credulous pipeline would publish this.",
    "",
    "FIVE-TEST GAUNTLET:",
    *[f"   {key:<16}: {val.upper()}"
      + ("   <-- SCANNER/SITE ARTIFACT" if key == "site_scanner" and val == "failed" else "")
      for key, val in k["robustness"].items()],
    "",
    f"robustness_score   : {k['robustness_score']} / 100",
    f"verdict            : {k['verdict'].upper()}",
    f"promoted           : {k['promoted']}",
    "",
    "BIOMARKER ANCHOR:",
    f"   status          : {kgate['status'].upper()}",
    f"   blocked_promotion: {kgate['blocked_promotion']}",
    "",
    f"honesty_rung CLAIMED : {card_kill.honesty_rung}   (NOT promoted)",
])


In [ ]:
# The card's own caveats say, in plain language, WHY it was killed.
show("ACT II  — why the referee refused to promote it", [
    *[textwrap.fill("- " + c, 72, subsequent_indent="  ")
      for c in k["caveats"][:4]],
])
assert k["promoted"] is False, "Act II must be KILLED (not promoted)"
assert k["robustness"]["site_scanner"] == "failed", "the scanner leak must fail"
print("\nAct II check: KILL not promoted; site_scanner test FAILED. The referee is honest. OK")


## The honest claim boundary

Two hypotheses, one instrument, two very different — and *defensible* — verdicts.

**What we are allowed to say about Act I:**

> The structural-embedding contrast is a **candidate** that survived the confound
> gauntlet and cleared the molecular-anchor gate on synthetic calibration data.
> Its honesty rung is **`severity_anchored`** — *ready for external replication*,
> **not** a proven biomarker. The ladder deliberately **caps here**: rung 5
> (`externally_replicated`) requires a genuinely independent second cohort, which
> a single-dataset run can never earn.

**What we refuse to say:**

The engine's honesty guard forbids the phrases "proven biomarker", "clinically
validated", "replaces PET", and similar overclaims — the run *raises* rather than
emit a card that trips them. Overclaiming is the cardinal sin; this instrument is
built so it structurally cannot.

**The named next experiment** (pre-registered above, not invented after the fact):
run the promoted contrast on **ADNI-3 / EPAD**, ~120 complete-case subjects with
plasma p-tau217, and **kill the routing** if the probe score shows no p-tau217
correlation (r < 0.2) on the complete-case subset.

That is the whole pitch: **a hypothesis-first instrument that pre-registers its
own falsifiers and is honest enough to kill its own findings.**
